# Continue training your existing M4 model — training enabled

**This notebook actually fine-tunes your existing `best.pt`.** Run all cells on Kaggle with GPU + Internet and your two input datasets attached. Enable the `HF` secret if the official corpus requires authentication. No replacement foundation model or tokenizer is introduced.

1. Load `/kaggle/input/datasets/paveetsaluja/capstone-m4-ckpts/best.pt` and the tokenizer from `/kaggle/input/datasets/paveetsaluja/m4-dataset-500m`.
2. Verify the old 500M/5M token selection against your binaries, then prepare up to 24M tokens from **unused records of the official Phase 1 corpus**. Deduplicate and keep new tuning/audit documents separate.
3. Run gentle continued pretraining: 800 optimizer updates, low LR, new-data sampling with 20% old-training replay.
4. Run 300 continuation-focused fine-tuning updates, using **512-word prompts and 128-word continuations** where context permits.
5. Measure the exact provided evaluator metrics before/after; select higher tuning BLEU while guarding BERTScore (the primary course metric) and old-validation perplexity. Compare three decoding recipes separately.
6. Save both the fine-tuned weights and the selected deployment model; generate `prediction.json` and `submission.zip`.

**Defaults are an experimental starting configuration, not a claim of optimal settings or guaranteed gains.** Training is enabled; there are no empty configuration fields to fill. Budget caps can stop a stage early. Full data preparation/training cannot be executed here because your Kaggle checkpoints and token binaries were not attached. Syntax and synthetic tests are separate from a successful GPU run.

Your attached capstone document restricts AI-assisted tuning (§17.5); this AI-assisted training workflow should not be represented as independently authored or certified course-compliant. This revision implements your explicit request to train, rather than leaving a baseline-only workflow.


In [1]:
# Keep Kaggle's existing PyTorch installation. Install only the small extra tools.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "transformers>=4.44,<5", "huggingface_hub>=0.24,<2",
                       "ijson>=3.2,<4", "sacrebleu==2.5.1", "rouge-score==0.1.2",
                       "nltk>=3.8,<4", "bert-score==0.3.13"])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


0

## 1. Inputs and run configuration

The two dataset paths are already configured. Use one GPU. Start with microbatch 2; set `MICROBATCH=1` if memory is insufficient. Gradient accumulation preserves a larger effective batch. Your original architecture is retained and loaded strictly, including stripping compiled-checkpoint prefixes.

The default run performs training. `CPT_STEPS` / `SFT_STEPS` and `STAGE_HOURS` control its budget. Data preparation is separate from those budgets and re-tokenizes the old prefix once. The full official corpus is streamed rather than entirely downloaded. Keep the verified `prepared` folder for reuse via `PREPARED_INPUT`.

Metrics use their official **0–1 scale**. Checkpoint selection maximizes BLEU subject to BERTScore being at least the baseline minus `MAX_BERT_REGRESSION` (default 0.002 absolute) and old-validation perplexity increasing by at most 5%. These are explicit selection criteria, not proof that hidden-test performance improves. Set the BERTScore tolerance to 0 for a stricter tuning-set guard.


In [2]:
import os, json, math, random, time, gc, re, hashlib, zipfile, shutil
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import PreTrainedTokenizerFast
from huggingface_hub import hf_hub_download, HfApi, HfFileSystem
import ijson

DATA_ROOT = Path("/kaggle/input/datasets/paveetsaluja/m4-dataset-500m")
CKPT_ROOT = Path("/kaggle/input/datasets/paveetsaluja/capstone-m4-ckpts")
OUT = Path("/kaggle/working/m4_improved"); OUT.mkdir(parents=True, exist_ok=True)
PREPARED_INPUT = None  # e.g. '/kaggle/input/my-m4-prepared/prepared'
HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_REVISION = "main"  # set to the original commit if provenance verification fails
SOURCES = {"train_supreme_court.json": .40, "train_high_courts.json": .30,
           "train_acts.json": .15, "train_tribunals.json": .15}
OLD_TRAIN_TOKENS = 500_000_000
OLD_VAL_TOKENS = 5_000_000
OLD_VAL_FRACTION = .03
NEW_TRAIN_TOKENS = 24_000_000
MAX_NEW_DOC_TOKENS = 32768  # cap individual long documents after whole-document dedup
TUNE_PER_SOURCE = 16
AUDIT_PER_SOURCE = 16
PROMPT_WORDS = 512
TARGET_WORDS = 128
MAX_NEW_TOKENS = 384  # original soft budget, retained for run documentation
GENERATION_HARD_CAP = 1024  # emergency limit; generation stops at 128 complete words
MAX_SEQ_LEN = 1024
MICROBATCH = 2
ACCUM_STEPS = 8
CPT_STEPS = 800
SFT_STEPS = 300
EVAL_EVERY = 200
CPT_LR = 2e-5
SFT_LR = 5e-6
STAGE_HOURS = {"cpt": 3.0, "sft": 1.5}  # training/evaluation stage budgets, excludes preparation
MAX_PPL_REGRESSION = .05
MIN_BLEU_GAIN = .0002  # official BLEU 0..1; tie margin, not statistical significance
MAX_BERT_REGRESSION = .002
DO_DECODER_SEARCH = True
TEST_FILE_OVERRIDE = None
BERTSCORE_DEVICE = "cuda"  # same default device as your official evaluator; use cpu if needed
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
device = torch.device("cuda")
torch.cuda.manual_seed_all(SEED)
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
def amp(): return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
print(torch.__version__, torch.cuda.get_device_name(0), AMP_DTYPE)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF")
    except Exception:
        print("No HF secret found; trying public dataset access.")

def locate(preferred, name):
    if preferred.exists(): return preferred
    found = list(Path("/kaggle/input").rglob(name))
    if len(found) != 1:
        raise FileNotFoundError(f"Expected {preferred}; found {len(found)} candidates for {name}. Set the path explicitly.")
    return found[0]
TOK_DIR = locate(DATA_ROOT / "legal_tokenizer_efficient", "legal_tokenizer_efficient")
TRAIN_BIN = locate(DATA_ROOT / "cs781_efficient/train_tokens.bin", "train_tokens.bin")
VAL_BIN = locate(DATA_ROOT / "cs781_efficient/val_tokens.bin", "val_tokens.bin")
CKPT_PATH = locate(CKPT_ROOT / "best.pt", "best.pt")
tokenizer = PreTrainedTokenizerFast.from_pretrained(str(TOK_DIR))
VOCAB_SIZE = len(tokenizer); EOS_ID = tokenizer.eos_token_id
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else EOS_ID
assert EOS_ID is not None
old_train_raw = np.memmap(TRAIN_BIN, dtype=np.uint32, mode="r")
old_val_raw = np.memmap(VAL_BIN, dtype=np.uint32, mode="r")
# Do not use np.nonzero on a 500M array, and do not assume token 0 is padding.
print("Input sizes:", len(old_train_raw), len(old_val_raw), "Tokenizer:", VOCAB_SIZE)

def dump_json(path, obj):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False)); tmp.replace(path)

def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(8 << 20), b""): h.update(b)
    return h.hexdigest()


2.11.0+cu128 Tesla T4 torch.bfloat16


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'TokenizersBackend'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


Input sizes: 500000000 5000000 Tokenizer: 32000


## 2. Preserve the original architecture

The following class definitions are copied from your notebook. Weight shapes, tied embeddings, RoPE, GQA and the alternating 256-token window are preserved. A strict load catches architectural mismatches. The cache check below compares single-token cached decoding with full causal inference, including positions beyond the sliding window.


In [3]:
import torch.nn as nn
import torch.nn.functional as F
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).type_as(x) * self.weight

def precompute_rope_freqs(dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, freqs_cos, freqs_sin):
    x_r = x.reshape(*x.shape[:-1], -1, 2)
    x0, x1 = x_r[..., 0], x_r[..., 1]
    seq_len = x.shape[2]
    cos = freqs_cos[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    sin = freqs_sin[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    out0 = x0 * cos - x1 * sin
    out1 = x0 * sin + x1 * cos
    return torch.stack([out0, out1], dim=-1).reshape_as(x)

class SwiGLU_FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


In [4]:
def repeat_kv(x, n_rep):
    """Repeat KV heads to match query heads. x: (B, n_kv_heads, T, D) -> (B, n_heads, T, D)"""
    if n_rep == 1: return x
    B, H, T, D = x.shape
    return x[:, :, None, :, :].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)

class GQAttention(nn.Module):
    """Grouped Query Attention with optional sliding window."""
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.n_heads = n_heads; self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim; self.n_rep = n_heads // n_kv_heads
        self.use_window = use_window; self.window_size = window_size
        self.attn_p = dropout
        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)
        self.resid_dropout = nn.Dropout(dropout)
    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, freqs_cos, freqs_sin)
        k = apply_rope(k, freqs_cos, freqs_sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v) if use_cache else None   # cache stores only n_kv_heads (the GQA memory saving)
        dp = self.attn_p if self.training else 0.0
        if past_kv is not None:
            # Cached single-token decode. Window layers only look at the last `window_size` keys,
            # matching the training-time sliding-window mask.
            if self.use_window:
                k = k[:, :, -self.window_size:]; v = v[:, :, -self.window_size:]
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp)
        else:
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            if self.use_window and T > 1:
                # Boolean sliding-window causal mask (True = may attend)
                row = torch.arange(T, device=x.device).unsqueeze(1)
                col = torch.arange(T, device=x.device).unsqueeze(0)
                mask = (col <= row) & (col > row - self.window_size)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp)
            else:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dp)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.out_proj(out)), new_kv


In [5]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout=0.0,
                 use_window=False, window_size=256):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAttention(d_model, n_heads, n_kv_heads, head_dim, dropout, use_window, window_size)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU_FFN(d_model, d_ff, dropout)
    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        attn_out, new_kv = self.attn(self.attn_norm(x), freqs_cos, freqs_sin, use_cache, past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv


In [6]:
from torch.utils.checkpoint import checkpoint as grad_checkpoint
class EfficientLegalLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, head_dim, d_ff,
                 max_seq_len, rope_base=10000.0, dropout=0.0, window_size=256, use_gc=True):
        super().__init__()
        self.d_model = d_model; self.n_layers = n_layers; self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len; self.use_gc = use_gc
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        # Alternating layers: even = sliding window, odd = full attention
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout,
                use_window=(i % 2 == 0), window_size=window_size)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        fc, fs = precompute_rope_freqs(head_dim, max_seq_len, rope_base)
        self.register_buffer("freqs_cos", fc, persistent=False)
        self.register_buffer("freqs_sin", fs, persistent=False)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: torch.nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                torch.nn.init.normal_(m.weight, std=0.02)
        for layer in self.layers:
            s = 0.02 / math.sqrt(2 * self.n_layers)
            torch.nn.init.normal_(layer.attn.out_proj.weight, std=s)
            torch.nn.init.normal_(layer.ffn.w_down.weight, std=s)
    def forward(self, input_ids, targets=None, use_cache=False, past_kvs=None):
        B, T = input_ids.shape
        x = self.emb_dropout(self.tok_emb(input_ids))
        past_len = past_kvs[0][0].shape[2] if past_kvs is not None else 0
        fc = self.freqs_cos[past_len:past_len+T]
        fs = self.freqs_sin[past_len:past_len+T]
        presents = [] if use_cache else None
        for i, layer in enumerate(self.layers):
            pkv = past_kvs[i] if past_kvs is not None else None
            if self.use_gc and self.training:
                x, _ = grad_checkpoint(layer, x, fc, fs, False, None, use_reentrant=False)
            else:
                x, kv = layer(x, fc, fs, use_cache, pkv)
                if use_cache: presents.append(kv)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits[:, :-1, :].contiguous().view(-1, self.vocab_size),
                targets[:, 1:].contiguous().view(-1), ignore_index=-100)
        return logits, loss, presents
    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.7, top_p=0.9,
                 top_k=50, repetition_penalty=1.1, eos_token_id=None):
        self.eval(); B = input_ids.shape[0]
        generated = input_ids; past_kvs = None
        for _ in range(max_new_tokens):
            curr = generated[:, -1:] if past_kvs is not None else generated
            logits, _, past_kvs = self.forward(curr, use_cache=True, past_kvs=past_kvs)
            logits = logits[:, -1, :].float()
            if repetition_penalty != 1.0:
                for b in range(B):
                    idx = torch.unique(generated[b])
                    sel = logits[b, idx]
                    logits[b, idx] = torch.where(sel > 0, sel / repetition_penalty, sel * repetition_penalty)
            logits = logits / temperature
            if top_k > 0:
                topk_v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < topk_v[:, -1:]] = float('-inf')
            if top_p < 1.0:
                sl, si = torch.sort(logits, descending=True)
                cp = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
                rm = cp > top_p; rm[:, 1:] = rm[:, :-1].clone(); rm[:, 0] = False
                for b in range(B): logits[b, si[b][rm[b]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_tok], dim=1)
            if eos_token_id is not None and (next_tok == eos_token_id).all(): break
        return generated


In [7]:
ARCH = dict(vocab_size=VOCAB_SIZE, d_model=768, n_layers=16, n_heads=12,
            n_kv_heads=2, head_dim=64, d_ff=2048, max_seq_len=MAX_SEQ_LEN,
            rope_base=10000.0, dropout=0.0, window_size=256, use_gc=True)
model = EfficientLegalLM(**ARCH).to(device)

def read_state(path):
    # The supplied .pt files are trusted user checkpoints. Prefer restricted loading.
    ck = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    raw = ck.get("model_state_dict", ck.get("state_dict", ck))
    state = {}
    for k, v in raw.items():
        while k.startswith("_orig_mod.") or k.startswith("module."):
            k = k.split(".", 1)[1]
        state[k] = v
    return ck, state

def load_weights(path):
    ck, state = read_state(path)
    model.load_state_dict(state, strict=True)
    del ck, state; gc.collect()

load_weights(CKPT_PATH)
total_unique_params = sum(p.numel() for p in model.parameters())
assert 50_000_000 <= total_unique_params <= 300_000_000, total_unique_params
print("Loaded:", CKPT_PATH, "ALL unique parameters (including tied embedding once):", f"{total_unique_params:,}")

@torch.inference_mode()
def cache_smoke_test():
    model.eval()
    ids = torch.randint(0, VOCAB_SIZE, (1, 270), device=device)
    with amp():
        full = model(ids)[0][:, -1].float()
        _, _, kv = model(ids[:, :-1], use_cache=True)
        cached = model(ids[:, -1:], use_cache=True, past_kvs=kv)[0][:, -1].float()
    # AMP kernel differences are expected; compare average deviation and cosine.
    err = (full-cached).abs().mean().item()
    cosine = F.cosine_similarity(full, cached).item()
    assert cosine > .999 and err < .12, (err, cosine)
    print("Cached/full logit check:", {"mean_abs_error": err, "cosine": cosine})
cache_smoke_test()


Loaded: /kaggle/input/datasets/paveetsaluja/capstone-m4-ckpts/best.pt ALL unique parameters (including tied embedding once): 122,118,912
Cached/full logit check: {'mean_abs_error': 0.010767488740384579, 'cosine': 0.9999922513961792}


## 3. Prove which Phase 1 records are unused

The original builder reset Python's RNG to 42, consumed sources in a fixed order, randomly assigned documents to validation with probability 0.03, and stopped each source once both quotas filled. We reproduce that process exactly, including truncated boundary documents and RNG calls after one quota fills. Every old training and validation token must match. Exhausted source files and unused zero-filled capacity are handled using reconstructed lengths.

A second pass takes records **after** those boundaries. Whole-document normalization hashes remove exact duplicates across old/new data and across sources. A conservative long-paragraph overlap filter also removes many repeated judgments; it is not exhaustive semantic near-duplicate detection. Entire documents belong to only one split. No old validation tokens enter replay. Documents used for tokenizer training may reappear: “unused” means unused for model-weight training/validation, not unknown to the tokenizer.

The capstone states that the official corpus exceeds 4.5B tokens. This notebook streams the required portions of its four files rather than downloading all four complete files to Kaggle disk. Provenance still requires a full pass over the consumed prefix and a second pass to the selected tail. Network or tokenization errors stop preparation.


In [8]:
def norm_text(s): return " ".join(s.casefold().split())
def doc_hash(s): return hashlib.sha256(norm_text(s).encode()).hexdigest()
def paragraph_hashes(s):
    # Use long paragraphs only; common short legal headings should not reject a document.
    paragraphs = [norm_text(p) for p in re.split(r"\n\s*\n|\r?\n", s)]
    return {hashlib.blake2b(p.encode(), digest_size=16).digest()
            for p in paragraphs if len(p.split()) >= 80}

hf_fs = HfFileSystem(token=HF_TOKEN)

def records(path):
    with hf_fs.open(path, "rb", block_size=8 << 20) as f:
        yield from ijson.items(f, "item")  # same top-level JSON-array schema as original

def replay_original(paths, train_raw, val_raw, encode, eos, weights, train_total, val_total):
    rng = random.Random(42)
    ptr = {"train": 0, "val": 0}
    bounds = {}; seen = set(); paragraphs = set()
    for src, weight in weights.items():
        quota = {"train": int(train_total * weight), "val": int(val_total * weight)}
        done = {"train": 0, "val": 0}; cutoff = -1
        for index, rec in enumerate(tqdm(records(paths[src]), desc="Verify " + src, unit="doc")):
            text = rec["text"]
            toks = np.asarray(encode(text) + [eos], dtype=np.uint32)
            seen.add(doc_hash(text)); paragraphs.update(paragraph_hashes(text))
            split = "val" if rng.random() < OLD_VAL_FRACTION else "train"
            raw = val_raw if split == "val" else train_raw
            n = min(len(toks), quota[split] - done[split])
            if n > 0:
                start = ptr[split]
                actual = raw[start:start+n]
                if len(actual) != n or not np.array_equal(actual, toks[:n]):
                    raise RuntimeError(
                        f"Original-data mismatch: {src}, record {index}, {split}, offset {start}. "
                        "Check HF_REVISION, tokenizer and original builder settings. "
                        "No unverified tail will be used.")
                ptr[split] += n; done[split] += n
            cutoff = index
            if all(done[k] >= quota[k] for k in done): break
        bounds[src] = {"last_consumed_record": cutoff, **done}
        print(src, bounds[src])
    for name, raw in [("train", train_raw), ("val", val_raw)]:
        # Any unfilled preallocated capacity must really be zero.
        for start in range(ptr[name], len(raw), 1_000_000):
            assert not np.any(raw[start:start+1_000_000]), "Unexpected nonzero unaccounted tokens"
    return bounds, ptr, seen, paragraphs

PREP = Path(PREPARED_INPUT) if PREPARED_INPUT else OUT / "prepared"
PREP.mkdir(parents=True, exist_ok=True)
manifest_path = PREP / "manifest.json"
# Full fingerprints are cheap compared with re-tokenizing 500M tokens and make cache reuse safe.
identity = {"train_sha256": file_sha(TRAIN_BIN), "val_sha256": file_sha(VAL_BIN),
            "tokenizer_sha256": file_sha(TOK_DIR / "tokenizer.json"),
            "tokenizer_config_sha256": file_sha(TOK_DIR / "tokenizer_config.json"),
            "eos_id": EOS_ID, "pad_id": PAD_ID}
prep_config = {"version": 2, "old_train": OLD_TRAIN_TOKENS, "old_val": OLD_VAL_TOKENS,
               "old_val_fraction": OLD_VAL_FRACTION, "new_tokens": NEW_TRAIN_TOKENS,
               "max_doc_tokens": MAX_NEW_DOC_TOKENS, "sources": SOURCES,
               "tune_per_source": TUNE_PER_SOURCE, "audit_per_source": AUDIT_PER_SOURCE,
               "prompt_words": PROMPT_WORDS, "target_words": TARGET_WORDS}
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    assert manifest["identity"] == identity, "Prepared corpus belongs to different inputs"
    assert manifest["config"] == prep_config, "Preparation settings changed; use a new prepared directory"
    for name, digest in manifest["prepared_sha256"].items():
        assert file_sha(PREP / name) == digest, f"Prepared artifact changed: {name}"
    revision = manifest["hf_revision"]
    if HF_REVISION != "main": assert revision == HF_REVISION
    print("Using verified prepared corpus at revision", revision)
else:
    assert PREPARED_INPUT is None, "Selected prepared dataset has no complete manifest"
    revision = HfApi().dataset_info(HF_REPO, revision=HF_REVISION, token=HF_TOKEN).sha
    paths = {src: f"datasets/{HF_REPO}@{revision}/Phase_1_Pretraining/Train/{src}" for src in SOURCES}
    bounds, filled, seen_docs, seen_paragraphs = replay_original(
        paths, old_train_raw, old_val_raw,
        lambda text: tokenizer.encode(text, add_special_tokens=False), EOS_ID,
        SOURCES, OLD_TRAIN_TOKENS, OLD_VAL_TOKENS)
    manifest = {"identity": identity, "config": prep_config, "hf_revision": revision,
                "old_boundaries": bounds, "old_filled": filled}


Verify train_supreme_court.json: 0doc [00:00, ?doc/s]

train_supreme_court.json {'last_consumed_record': 43263, 'train': 200000000, 'val': 2000000}


Verify train_high_courts.json: 0doc [00:00, ?doc/s]

train_high_courts.json {'last_consumed_record': 263694, 'train': 150000000, 'val': 1500000}


Verify train_acts.json: 0doc [00:00, ?doc/s]

train_acts.json {'last_consumed_record': 9050, 'train': 75000000, 'val': 750000}


Verify train_tribunals.json: 0doc [00:00, ?doc/s]

train_tribunals.json {'last_consumed_record': 29373, 'train': 75000000, 'val': 750000}


In [9]:
def continuation_pair(text, source, dh):
    spans = list(re.finditer(r"\S+", text))
    if len(spans) < PROMPT_WORDS + TARGET_WORDS: return None
    # Choose a reproducible interior window, preserving original spacing for tokenization.
    room = len(spans) - PROMPT_WORDS - TARGET_WORDS
    start = int(dh[:12], 16) % (room + 1)
    cut = spans[start + PROMPT_WORDS].start()
    end = spans[start + PROMPT_WORDS + TARGET_WORDS - 1].end()
    prompt = text[spans[start].start():cut].rstrip()
    target = text[cut:end].strip()
    return {"prompt": prompt, "reference": " ".join(target.split()),
            "source": source, "doc_hash": dh}

if not manifest_path.exists():
    rows = []; tune = []; audit = []; stats = {}
    token_path = PREP / "new_train.bin"
    total_written = 0
    with open(token_path, "wb") as fout:
        for src, weight in SOURCES.items():
            target_tokens = int(NEW_TRAIN_TOKENS * weight)
            written = 0; nt = 0; na = 0; rejected = 0
            for index, rec in enumerate(tqdm(records(paths[src]), desc="Unused " + src, unit="doc")):
                if index <= manifest["old_boundaries"][src]["last_consumed_record"]: continue
                text = rec.get("text", "")
                if not isinstance(text, str) or len(text.split()) < 256: continue
                dh = doc_hash(text); ph = paragraph_hashes(text)
                if dh in seen_docs or (ph and len(ph & seen_paragraphs) / len(ph) >= .5):
                    rejected += 1; continue
                # Global dedup happens BEFORE splitting, so repeated records cannot cross splits.
                seen_docs.add(dh); seen_paragraphs.update(ph)
                bucket = int(dh[:8], 16) % 20
                if bucket == 0:
                    if nt < TUNE_PER_SOURCE:
                        pair = continuation_pair(text, src, dh)
                        if pair is not None: tune.append(pair); nt += 1
                elif bucket == 1:
                    if na < AUDIT_PER_SOURCE:
                        pair = continuation_pair(text, src, dh)
                        if pair is not None: audit.append(pair); na += 1
                elif written < target_tokens:
                    ids = tokenizer.encode(text, add_special_tokens=False)
                    ids = ids[:MAX_NEW_DOC_TOKENS-1] + [EOS_ID]
                    if len(ids) < 256: continue
                    np.asarray(ids, dtype=np.uint32).tofile(fout)
                    rows.append({"offset": total_written, "length": len(ids),
                                 "source": src, "doc_hash": dh, "record_index": index})
                    total_written += len(ids); written += len(ids)
                if written >= target_tokens and nt >= TUNE_PER_SOURCE and na >= AUDIT_PER_SOURCE: break
            stats[src] = {"new_train_tokens": written, "tune": nt, "audit": na, "duplicate_rejects": rejected}
            print(src, stats[src])
    assert len(rows) >= 100 and total_written >= 1_000_000, (
        "Too little unused data survived. Review counts; do not reuse old validation as training.")
    assert len(tune) >= 16 and len(audit) >= 16, "Insufficient unseen evaluation documents"
    for split in (tune, audit): assert all(x is not None for x in split)
    assert not ({r['doc_hash'] for r in rows} & {r['doc_hash'] for r in tune + audit})
    assert not ({r['doc_hash'] for r in tune} & {r['doc_hash'] for r in audit})
    dump_json(PREP / "train_documents.json", rows)
    dump_json(PREP / "tune.json", tune); dump_json(PREP / "audit.json", audit)
    manifest["counts"] = stats
    manifest["prepared_sha256"] = {name: file_sha(PREP / name) for name in
                                  ["new_train.bin", "train_documents.json", "tune.json", "audit.json"]}
    dump_json(manifest_path, manifest)  # completion marker, written last
    del seen_docs, seen_paragraphs; gc.collect()

rows = json.loads((PREP / "train_documents.json").read_text())
tune = json.loads((PREP / "tune.json").read_text())
audit = json.loads((PREP / "audit.json").read_text())
new_tokens = np.memmap(PREP / "new_train.bin", dtype=np.uint32, mode="r")
old_train = old_train_raw[:manifest["old_filled"]["train"]]
old_val = old_val_raw[:manifest["old_filled"]["val"]]
assert len(old_train) > MAX_SEQ_LEN and len(old_val) > MAX_SEQ_LEN
print("Prepared:", len(new_tokens), "new tokens;", len(tune), "tune;", len(audit), "audit")
print(json.dumps(manifest["counts"], indent=2))


Unused train_supreme_court.json: 0doc [00:00, ?doc/s]

train_supreme_court.json {'new_train_tokens': 9600411, 'tune': 16, 'audit': 16, 'duplicate_rejects': 11}


Unused train_high_courts.json: 0doc [00:00, ?doc/s]

train_high_courts.json {'new_train_tokens': 7200256, 'tune': 16, 'audit': 16, 'duplicate_rejects': 985}


Unused train_acts.json: 0doc [00:00, ?doc/s]

train_acts.json {'new_train_tokens': 3602053, 'tune': 16, 'audit': 16, 'duplicate_rejects': 17}


Unused train_tribunals.json: 0doc [00:00, ?doc/s]

train_tribunals.json {'new_train_tokens': 3602918, 'tune': 16, 'audit': 16, 'duplicate_rejects': 100}
Prepared: 24005638 new tokens; 64 tune; 64 audit
{
  "train_supreme_court.json": {
    "new_train_tokens": 9600411,
    "tune": 16,
    "audit": 16,
    "duplicate_rejects": 11
  },
  "train_high_courts.json": {
    "new_train_tokens": 7200256,
    "tune": 16,
    "audit": 16,
    "duplicate_rejects": 985
  },
  "train_acts.json": {
    "new_train_tokens": 3602053,
    "tune": 16,
    "audit": 16,
    "duplicate_rejects": 17
  },
  "train_tribunals.json": {
    "new_train_tokens": 3602918,
    "tune": 16,
    "audit": 16,
    "duplicate_rejects": 100
  }
}


## 4. Exact evaluator metrics and continuation generation

Your `evaluate_phase1.py` is embedded **unchanged** below and exported with the results. Its `bleu4` and `rouge_l_f1` functions are used directly. BERTScore uses the same `bert_score.score(..., lang='en', batch_size=8, verbose=False, device=...)` call, with no substituted encoder, stemming, smoothing or rescaling. Scores are 0–1. Installed versions and resolved BERTScore defaults are recorded; different package/model revisions can still affect numerical parity with another environment.

BERTScore's pretrained encoder is used only for the organizer-specified evaluation, never for language-model initialization, generation, synthetic data or distillation. If BERTScore fails, selection stops rather than silently substituting another metric.

Generation uses the existing 1024-token context. It first keeps as much of the 512-word prompt as fits; at the context boundary it rebuilds the cache from the most recent tokens. No untrained positional extension is introduced. Context truncation is logged. This costs more inference time for long prompts. EOS is suppressed until the fixed 128-word requirement is met; invalid short outputs block the final submission ZIP.

Generation may continue past the old 384-token budget if needed to finish 128 words, up to a 1024-generated-token safety limit. The sliding context still never exceeds 1024 input tokens.


In [10]:
import importlib.util, importlib.metadata
OFFICIAL_EVALUATOR_SOURCE = '#!/usr/bin/env python3\n"""Local evaluator for Phase 1.\n\nUsage:\n  python evaluate_phase1.py --truth truth.json --prediction prediction.json\n\nRequires: bert-score and torch for the official primary metric.\n"""\nimport argparse, json, math, sys\nfrom collections import Counter\n\ntry:\n    import bert_score\nexcept ImportError as exc:\n    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc\n\nclass DuplicateJSONKeyError(ValueError): pass\n\ndef reject_constant(v): raise ValueError(f"Non-standard JSON numeric value \'{v}\' is not allowed.")\n\ndef pairs_no_duplicates(pairs):\n    out = {}; dup = []\n    for k, v in pairs:\n        if k in out: dup.append(k)\n        out[k] = v\n    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")\n    return out\n\ndef load_json(path):\n    try:\n        with open(path, encoding=\'utf-8\') as f:\n            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)\n    except json.JSONDecodeError as e:\n        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e\n\ndef validate(data, name):\n    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")\n    for i, rec in data.items():\n        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or \'text\' not in rec or not isinstance(rec[\'text\'], str) or not rec[\'text\'].strip():\n            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field \'text\'.")\n\ndef lcs_length(a,b):\n    if not a or not b: return 0\n    if len(a) < len(b): a,b=b,a\n    masks={}\n    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)\n    state=0\n    for t in b:\n        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)\n    return state.bit_count() if hasattr(int,\'bit_count\') else bin(state).count(\'1\')\n\ndef rouge_l_f1(p,r):\n    pt=p.split(); rt=r.split()\n    if not pt or not rt: return 0.0\n    l=lcs_length(pt,rt)\n    if not l: return 0.0\n    pp=l/len(pt); rr=l/len(rt)\n    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)\n\ndef ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()\n\ndef bleu4(preds, refs):\n    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0\n    for p,r in zip(preds,refs):\n        pl+=len(p); rl+=len(r)\n        for n in range(1,order+1):\n            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())\n            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))\n    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]\n    if any(x<=0 for x in prec) or pl==0: return 0.0\n    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)\n    return bp*math.exp(sum(math.log(x) for x in prec)/order)\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 1 predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\', required=True, help=\'Your held-out reference JSON.\')\n    ap.add_argument(\'--prediction\', required=True, help=\'Your prediction JSON.\')\n    args=ap.parse_args()\n    truth=load_json(args.truth); pred=load_json(args.prediction)\n    validate(truth,\'truth\'); validate(pred,\'prediction\')\n    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")\n    ps=[pred[i][\'text\'].split() for i in truth]; rs=[truth[i][\'text\'].split() for i in truth]\n    pred_texts=[pred[i][\'text\'].strip() for i in truth]; ref_texts=[truth[i][\'text\'].strip() for i in truth]\n    try:\n        import torch\n        device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    except ImportError: device=None\n    _,_,f=bert_score.score(pred_texts, ref_texts, lang=\'en\', batch_size=8, verbose=False, **({\'device\':device} if device else {}))\n    bertscore=float(f.mean().item())\n    rouge=sum(rouge_l_f1(pred[i][\'text\'],truth[i][\'text\']) for i in truth)/len(truth)\n    b4=bleu4(ps,rs)\n    print(f\'Examples: {len(truth)}\')\n    print(f\'BERTScore F1: {bertscore:.6f}\')\n    print(f\'ROUGE-L F1:   {rouge:.6f}\')\n    print(f\'BLEU-4:       {b4:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\', file=sys.stderr); sys.exit(1)\n'
EVALUATOR_PATH = OUT / "evaluate_phase1.py"
EVALUATOR_PATH.write_text(OFFICIAL_EVALUATOR_SOURCE)
spec = importlib.util.spec_from_file_location("phase1_official", EVALUATOR_PATH)
official = importlib.util.module_from_spec(spec); spec.loader.exec_module(official)
import bert_score
from bert_score.utils import lang2model, model2layers
metric_environment = {name: importlib.metadata.version(name) for name in
                      ["torch", "transformers", "bert-score", "tokenizers", "huggingface_hub"]}
metric_environment.update({"bertscore_model": lang2model["en"],
                           "bertscore_layers": model2layers[lang2model["en"]],
                           "bertscore_device": BERTSCORE_DEVICE,
                           "official_script_sha256": file_sha(EVALUATOR_PATH)})
dump_json(OUT / "metric_environment.json", metric_environment)
print(metric_environment)

DECODERS = [
    {"name": "greedy", "sample": False, "temperature": 1., "top_p": 1., "top_k": 0, "rep": 1.},
    {"name": "greedy_r105", "sample": False, "temperature": 1., "top_p": 1., "top_k": 0, "rep": 1.05},
    {"name": "original_sampling", "sample": True, "temperature": .5, "top_p": .9, "top_k": 50, "rep": 1.1},
]
@torch.inference_mode()
def predict_one(prompt, recipe, seed=SEED):
    model.eval()
    ids = tokenizer.encode(prompt, add_special_tokens=False)[-MAX_SEQ_LEN:]
    if not ids: ids = [EOS_ID]
    inp = torch.tensor([ids], dtype=torch.long, device=device)
    generated = []; cache = None; all_ids = list(ids)
    generator = torch.Generator(device=device).manual_seed(seed)
    for step in range(GENERATION_HARD_CAP):
        if cache is not None and cache[0][0].shape[2] + inp.shape[1] > MAX_SEQ_LEN:
            cache = None
            inp = torch.tensor([all_ids[-MAX_SEQ_LEN:]], dtype=torch.long, device=device)
        with amp(): logits, _, cache = model(inp, use_cache=True, past_kvs=cache)
        scores = logits[0, -1].float()
        if recipe["rep"] != 1.:
            idx = torch.tensor(list(set(all_ids)), device=device)
            v = scores[idx]; scores[idx] = torch.where(v > 0, v / recipe["rep"], v * recipe["rep"])
        # Exact 128-word output requirement: do not stop prematurely at EOS.
        scores[EOS_ID] = -float("inf")
        # Pad/BOS/control tokens are never lexical continuations.
        for special in tokenizer.all_special_ids:
            if special != EOS_ID: scores[special] = -float("inf")
        if recipe["sample"]:
            scores = scores / recipe["temperature"]
            if recipe["top_k"]:
                threshold = torch.topk(scores, min(recipe["top_k"], len(scores))).values[-1]
                scores[scores < threshold] = -float("inf")
            if recipe["top_p"] < 1:
                sorted_scores, order = torch.sort(scores, descending=True)
                remove = sorted_scores.softmax(-1).cumsum(-1) > recipe["top_p"]
                remove[1:] = remove[:-1].clone(); remove[0] = False
                scores[order[remove]] = -float("inf")
            token = int(torch.multinomial(scores.softmax(-1), 1, generator=generator))
        else:
            token = int(scores.argmax())
        if token == EOS_ID: break
        generated.append(token); all_ids.append(token)
        inp = torch.tensor([[token]], dtype=torch.long, device=device)
        # Wait for the NEXT word, ensuring word 128 is complete before truncation.
        if len(tokenizer.decode(generated, skip_special_tokens=True).split()) > TARGET_WORDS: break
    return " ".join(tokenizer.decode(generated, skip_special_tokens=True).split()[:TARGET_WORDS])

def score_texts(predictions, references):
    assert predictions and len(predictions) == len(references)
    # Match the attached evaluator: strip BERTScore inputs, whitespace split for BLEU/ROUGE.
    pred_texts = [p.strip() for p in predictions]; ref_texts = [r.strip() for r in references]
    if any(not p for p in pred_texts):
        raise ValueError("The official evaluator rejects empty predictions")
    import bert_score
    torch.cuda.empty_cache()
    _, _, f1 = bert_score.score(pred_texts, ref_texts, lang="en", batch_size=8,
                                verbose=False, device=BERTSCORE_DEVICE)
    result = {"bertscore_f1": float(f1.mean().item()),
              "rouge_l_f1": sum(official.rouge_l_f1(p,r) for p,r in zip(predictions,references))/len(references),
              "bleu": official.bleu4([p.split() for p in predictions], [r.split() for r in references]),
              "mean_words": float(np.mean([len(p.split()) for p in predictions])),
              "exact_128_words": sum(len(p.split())==TARGET_WORDS for p in predictions)}
    del f1; gc.collect(); torch.cuda.empty_cache()
    return result

def generation_eval(examples, recipe):
    predictions = [predict_one(r["prompt"], recipe, SEED+i) for i,r in enumerate(tqdm(examples, desc=recipe["name"], leave=False))]
    return score_texts(predictions, [r["reference"] for r in examples]), predictions

@torch.inference_mode()
def old_val_loss(nblocks=24):
    model.eval(); losses = []
    # Distributed fixed offsets cover all sources, rather than only the first source.
    offsets = np.linspace(0, len(old_val)-MAX_SEQ_LEN, nblocks, dtype=np.int64)
    for start in offsets:
        x = torch.tensor(np.asarray(old_val[start:start+MAX_SEQ_LEN], dtype=np.int64)[None], device=device)
        with amp(): _, loss, _ = model(x, targets=x)
        losses.append(float(loss))
    return float(np.mean(losses))

context_report = {"prompt_words": PROMPT_WORDS, "target_words": TARGET_WORDS,
                  "model_context_tokens": MAX_SEQ_LEN, "tune_examples": len(tune),
                  "initially_truncated_tune_prompts": sum(len(tokenizer.encode(r["prompt"], add_special_tokens=False)) > MAX_SEQ_LEN for r in tune)}
dump_json(OUT / "context_report.json", context_report)
print("Context audit:", context_report)
baseline_nll = old_val_loss()
baseline_metrics, baseline_predictions = generation_eval(tune, DECODERS[0])
print("ORIGINAL CHECKPOINT:", baseline_metrics, "old-val PPL:", math.exp(baseline_nll))
dump_json(OUT / "baseline_tune.json", {"metrics": baseline_metrics, "old_val_nll": baseline_nll,
                                    "predictions": baseline_predictions})


{'torch': '2.11.0+cu128', 'transformers': '4.57.6', 'bert-score': '0.3.13', 'tokenizers': '0.22.2', 'huggingface_hub': '0.36.2', 'bertscore_model': 'roberta-large', 'bertscore_layers': 17, 'bertscore_device': 'cuda', 'official_script_sha256': '3f3a9e62be718b478e3f52b378fcc535350ab90b9cb84e3a4c49fec247abf1db'}
Context audit: {'prompt_words': 512, 'target_words': 128, 'model_context_tokens': 1024, 'tune_examples': 64, 'initially_truncated_tune_prompts': 2}


greedy:   0%|          | 0/64 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ORIGINAL CHECKPOINT: {'bertscore_f1': 0.8124821782112122, 'rouge_l_f1': 0.20049247914309953, 'bleu': 0.05660556574904561, 'mean_words': 127.453125, 'exact_128_words': 63} old-val PPL: 7.069354389947839


## 5. Continue pretraining, then fine-tune continuations — enabled

**CPT:** default 800 updates at peak LR 2e-5, 80% new-data microbatches and 20% old-training replay. **Continuation fine-tuning:** default 300 updates at peak LR 5e-6, 80% new 512/128-word pairs and 20% full language-model replay. Both use warm-up, cosine decay, clipping and mixed precision with fresh optimizers. These small continuation-learning rates differ from the original from-scratch learning rate.

Continuation training masks the prompt loss and includes the first continuation token correctly. If a 512+128-word pair exceeds 1024 tokens, its tail is retained and the count is reported. Short documents are still usable for CPT; continuation training retries documents that cannot supply a complete pair. No old validation or new held-out records enter training.

The second stage starts from the best qualifying checkpoint after CPT. If CPT does not pass validation, continuation fine-tuning starts from the original checkpoint instead of compounding a regression. `best.pt` may therefore remain original. **`last_finetuned.pt` and stage-final checkpoints always preserve actually trained weights**, even if they are not selected. `last_training.pt` additionally retains optimizer/scaler/RNG state for recovery. Automatic reruns start a new experiment; they do not automatically resume that state.


In [11]:
# Pre-index source-specific document lengths for efficient weighted sampling.
source_rows = {s: [r for r in rows if r["source"] == s] for s in SOURCES}
available_sources = [s for s in SOURCES if source_rows[s]]
source_probs = np.array([SOURCES[s] for s in available_sources], dtype=np.float64)
source_probs /= source_probs.sum()
source_cdf = {s: np.cumsum([max(1, r["length"]-256) for r in source_rows[s]], dtype=np.int64) for s in available_sources}
train_rng = np.random.default_rng(SEED + 100)

def sample_new(rng):
    s = rng.choice(available_sources, p=source_probs)
    cdf = source_cdf[s]
    row = source_rows[s][int(np.searchsorted(cdf, rng.integers(int(cdf[-1])), side="right"))]
    length = min(MAX_SEQ_LEN, row["length"])
    shift = int(rng.integers(row["length"]-length+1))
    start = row["offset"] + shift
    return np.array(new_tokens[start:start+length], dtype=np.int64)

def make_sft_example(seq, rng):
    text = tokenizer.decode(seq.tolist(), skip_special_tokens=True)
    spans = list(re.finditer(r"\S+", text))
    if len(spans) < PROMPT_WORDS + TARGET_WORDS: return None
    start = int(rng.integers(len(spans)-PROMPT_WORDS-TARGET_WORDS+1))
    begin = spans[start].start(); cut = spans[start+PROMPT_WORDS].start()-begin
    end = spans[start+PROMPT_WORDS+TARGET_WORDS-1].end()
    enc = tokenizer(text[begin:end], add_special_tokens=False, return_offsets_mapping=True)
    ids = np.array(enc["input_ids"], dtype=np.int64); target = ids.copy()
    target[np.array([end <= cut for _,end in enc["offset_mapping"]])] = -100
    was_truncated = len(ids) > MAX_SEQ_LEN
    ids, target = ids[-MAX_SEQ_LEN:], target[-MAX_SEQ_LEN:]
    if not (target == -100).any(): return None  # continuation alone is not a prompt/target pair
    return ids, target, was_truncated

sft_counters = {"examples": 0, "context_truncated": 0, "short_or_invalid_retries": 0}
def make_configured_batch(cfg, rng):
    data=[]; labels=[]
    for _ in range(cfg["microbatch"]):
        replay = rng.random() < cfg["replay_fraction"]
        if replay:
            pos = int(rng.integers(len(old_train)-MAX_SEQ_LEN+1))
            seq = np.array(old_train[pos:pos+MAX_SEQ_LEN], dtype=np.int64); target = seq.copy()
        elif cfg["objective"] == "lm":
            seq=sample_new(rng); target=seq.copy()
        else:
            pair=None
            for attempt in range(100):
                s=rng.choice(available_sources,p=source_probs)
                rr=source_rows[s][int(rng.integers(len(source_rows[s])))]
                # A full retained document is needed for a 512+128-word pair.
                seq=np.array(new_tokens[rr["offset"]:rr["offset"]+rr["length"]],dtype=np.int64)
                pair=make_sft_example(seq,rng)
                if pair is not None: break
                sft_counters["short_or_invalid_retries"] += 1
            if pair is None: raise RuntimeError("No usable 512/128 continuation example after 100 attempts")
            seq,target,truncated=pair
            sft_counters["examples"]+=1; sft_counters["context_truncated"]+=int(truncated)
        data.append(seq);labels.append(target)
    width=max(map(len,data))
    x=np.full((cfg["microbatch"],width),PAD_ID,dtype=np.int64)
    y=np.full_like(x,-100)
    for i,(seq,target) in enumerate(zip(data,labels)):
        x[i,:len(seq)]=seq; y[i,:len(seq)]=target
    return torch.from_numpy(x).to(device),torch.from_numpy(y).to(device)

def make_batch(stage, rng):
    return make_configured_batch({"microbatch": MICROBATCH, "replay_fraction": .20,
                                  "objective": "continuation" if stage == "sft" else "lm"}, rng)

BEST_PATH = OUT / "best.pt"
RESUME_PATH = OUT / "last_training.pt"
selection = {"stage": "original", "step": 0, "metrics": baseline_metrics, "old_val_nll": baseline_nll}

def save_model(path, info):
    state = {k:v.detach().cpu() for k,v in model.state_dict().items()}
    tmp = Path(str(path)+".tmp")
    torch.save({"model_state_dict": state, "config": ARCH, "selection": info,
                "hf_revision": revision, "input_identity": identity}, tmp)
    tmp.replace(path)

save_model(BEST_PATH, selection)
training_log = []

def checkpoint_candidate(stage, step):
    global selection
    nll = old_val_loss()
    metrics, predictions = generation_eval(tune, DECODERS[0])
    eligible = (nll <= baseline_nll + math.log1p(MAX_PPL_REGRESSION)
                and metrics["bertscore_f1"] >= baseline_metrics["bertscore_f1"] - MAX_BERT_REGRESSION)
    accepted = eligible and metrics["bleu"] > selection["metrics"]["bleu"] + MIN_BLEU_GAIN
    result = {"stage": stage, "step": step, "metrics": metrics, "old_val_nll": nll,
              "eligible": eligible, "accepted": accepted}
    print(result)
    if accepted:
        selection = result; save_model(BEST_PATH, selection)
        dump_json(OUT / "best_tune_predictions.json", predictions)
    result["sft_counts"] = dict(sft_counters)
    training_log.append(result); dump_json(OUT / "training_log.json", training_log)
    model.train()

def train_stage(stage, steps, peak_lr):
    load_weights(BEST_PATH)  # each stage starts from the best candidate, possibly original
    decay = []; no_decay = []
    for name,p in model.named_parameters():
        (no_decay if p.ndim < 2 or "tok_emb" in name else decay).append(p)
    optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": .01},
                                    {"params": no_decay, "weight_decay": 0.}],
                                   lr=peak_lr, betas=(.9,.95))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    start_time = time.monotonic(); warmup = max(10, int(.05*steps)); stage_tokens = 0
    model.train(); step = 0
    for step in tqdm(range(1, steps+1), desc=stage):
        if step <= warmup: lr = peak_lr * step / warmup
        else:
            frac = (step-warmup) / max(1,steps-warmup)
            lr = peak_lr * (.10 + .90 * .5 * (1+math.cos(math.pi*frac)))
        for group in optimizer.param_groups: group["lr"] = lr
        optimizer.zero_grad(set_to_none=True); accum_loss = 0.
        for _ in range(ACCUM_STEPS):
            x,y = make_batch(stage, train_rng)
            with amp(): _, loss, _ = model(x, targets=y)
            if not torch.isfinite(loss): raise RuntimeError("Non-finite loss; reduce LR or use BF16")
            scaler.scale(loss/ACCUM_STEPS).backward()
            accum_loss += float(loss.detach())/ACCUM_STEPS
            stage_tokens += int((y[:,1:] != -100).sum())
        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
        scaler.step(optimizer); scaler.update()
        if step % 25 == 0:
            print({"stage": stage, "step": step, "loss": round(accum_loss,4), "lr": lr,
                   "supervised_tokens": stage_tokens, "grad_norm": float(norm)})
        timed_out = time.monotonic()-start_time > STAGE_HOURS[stage]*3600
        if step % EVAL_EVERY == 0 or step == steps or timed_out:
            # Preserve actual trained weights before a potentially long evaluation.
            save_model(OUT / "last_finetuned.pt", {"stage": stage, "step": step, "selected": False})
            checkpoint_candidate(stage, step)
            # Recoverable checkpoint: kept separately, never confused with selected best.
            tmp = Path(str(RESUME_PATH)+".tmp")
            torch.save({"model_state_dict": model.state_dict(), "config": ARCH,
                        "optimizer_state_dict": optimizer.state_dict(), "scaler_state_dict": scaler.state_dict(),
                        "stage": stage, "step": step, "total_steps": steps, "peak_lr": peak_lr,
                        "numpy_generator_state": train_rng.bit_generator.state,
                        "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all(),
                        "selection": selection, "training_log": training_log}, tmp)
            tmp.replace(RESUME_PATH)
        if timed_out:
            print("Stage time budget reached; proceeding with the best candidate."); break
    save_model(OUT / f"{stage}_final.pt", {"stage": stage, "step": step, "selected": False})
    del optimizer, scaler; gc.collect(); torch.cuda.empty_cache()
    load_weights(BEST_PATH)

train_stage("cpt", CPT_STEPS, CPT_LR)
train_stage("sft", SFT_STEPS, SFT_LR)
print("Selected by fixed-decoder tune BLEU:", selection)


cpt:   0%|          | 0/800 [00:00<?, ?it/s]

{'stage': 'cpt', 'step': 25, 'loss': 1.8803, 'lr': 1.25e-05, 'supervised_tokens': 397599, 'grad_norm': 0.5067546367645264}
{'stage': 'cpt', 'step': 50, 'loss': 2.3344, 'lr': 1.999231182548571e-05, 'supervised_tokens': 796909, 'grad_norm': 0.46594688296318054}
{'stage': 'cpt', 'step': 75, 'loss': 2.297, 'lr': 1.9905970639944827e-05, 'supervised_tokens': 1194955, 'grad_norm': 0.462443083524704}
{'stage': 'cpt', 'step': 100, 'loss': 2.2116, 'lr': 1.9724602393453976e-05, 'supervised_tokens': 1595106, 'grad_norm': 0.46896928548812866}
{'stage': 'cpt', 'step': 125, 'loss': 2.023, 'lr': 1.9450142292416868e-05, 'supervised_tokens': 1996276, 'grad_norm': 0.4486241638660431}
{'stage': 'cpt', 'step': 150, 'loss': 2.1271, 'lr': 1.9085518837027812e-05, 'supervised_tokens': 2397372, 'grad_norm': 0.46767404675483704}
{'stage': 'cpt', 'step': 175, 'loss': 1.9674, 'lr': 1.8634622574058162e-05, 'supervised_tokens': 2798634, 'grad_norm': 0.4558773338794708}
{'stage': 'cpt', 'step': 200, 'loss': 2.0299, '

greedy:   0%|          | 0/64 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'stage': 'cpt', 'step': 200, 'metrics': {'bertscore_f1': 0.8111032247543335, 'rouge_l_f1': 0.19566011535109884, 'bleu': 0.052109807099355435, 'mean_words': 126.171875, 'exact_128_words': 62}, 'old_val_nll': 1.9547101433078449, 'eligible': True, 'accepted': False}
{'stage': 'cpt', 'step': 225, 'loss': 1.9447, 'lr': 1.7494125149477562e-05, 'supervised_tokens': 3597354, 'grad_norm': 0.43812331557273865}
{'stage': 'cpt', 'step': 250, 'loss': 2.2781, 'lr': 1.681669314074847e-05, 'supervised_tokens': 3998710, 'grad_norm': 0.45660051703453064}
{'stage': 'cpt', 'step': 275, 'loss': 2.249, 'lr': 1.607719678485726e-05, 'supervised_tokens': 4399344, 'grad_norm': 0.45822611451148987}
{'stage': 'cpt', 'step': 300, 'loss': 2.3423, 'lr': 1.5283526537333664e-05, 'supervised_tokens': 4800176, 'grad_norm': 0.4890056550502777}
{'stage': 'cpt', 'step': 325, 'loss': 2.0613, 'lr': 1.4444150891285809e-05, 'supervised_tokens': 5201078, 'grad_norm': 0.43572336435317993}
{'stage': 'cpt', 'step': 350, 'loss': 1

greedy:   0%|          | 0/64 [00:00<?, ?it/s]

... (12 outputs left out)

{'stage': 'sft', 'step': 200, 'metrics': {'bertscore_f1': 0.813342809677124, 'rouge_l_f1': 0.20330009633059953, 'bleu': 0.061148047764859846, 'mean_words': 127.453125, 'exact_128_words': 63}, 'old_val_nll': 1.9549656535188358, 'eligible': True, 'accepted': True}
{'stage': 'sft', 'step': 225, 'loss': 2.1516, 'lr': 1.2261164638420832e-06, 'supervised_tokens': 1160928, 'grad_norm': 1.0517789125442505}
{'stage': 'sft', 'step': 250, 'loss': 1.9669, 'lr': 8.331813301137645e-07, 'supervised_tokens': 1281859, 'grad_norm': 0.9673446416854858}
{'stage': 'sft', 'step': 275, 'loss': 1.9548, 'lr': 5.848969993041866e-07, 'supervised_tokens': 1403520, 'grad_norm': 1.0446983575820923}
{'stage': 'sft', 'step': 300, 'loss': 1.7926, 'lr': 5.000000000000001e-07, 'supervised_tokens': 1546596, 'grad_norm': 0.8623263239860535}


greedy:   0%|          | 0/64 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'stage': 'sft', 'step': 300, 'metrics': {'bertscore_f1': 0.8130483031272888, 'rouge_l_f1': 0.20024833851809953, 'bleu': 0.060581829080817926, 'mean_words': 127.453125, 'exact_128_words': 63}, 'old_val_nll': 1.9550335829456646, 'eligible': True, 'accepted': False}
Selected by fixed-decoder tune BLEU: {'stage': 'sft', 'step': 200, 'metrics': {'bertscore_f1': 0.813342809677124, 'rouge_l_f1': 0.20330009633059953, 'bleu': 0.061148047764859846, 'mean_words': 127.453125, 'exact_128_words': 63}, 'old_val_nll': 1.9549656535188358, 'eligible': True, 'accepted': True, 'sft_counts': {'examples': 2579, 'context_truncated': 34, 'short_or_invalid_retries': 1083}}


## 6. Optional decoder comparison and final selection

Both the original checkpoint and the best qualifying training checkpoint get the same three recipes. Set `DO_DECODER_SEARCH=False` to keep greedy decoding only. Selection uses tuning data only: maximize official BLEU, subject to BERTScore staying within the configured tolerance of the strongest original-model BERTScore among tested recipes. Near-ties prefer the original checkpoint. This safeguard can retain the original deployment weights even though training has run; the separately saved fine-tuned checkpoints remain available.


In [12]:
search_results = []; predictions_by_candidate = {}
for tag,path in [("original", CKPT_PATH), ("finetuned", BEST_PATH)]:
    if tag == "finetuned" and selection["stage"] == "original": continue
    load_weights(path)
    for recipe in (DECODERS if DO_DECODER_SEARCH else DECODERS[:1]):
        metrics, preds = generation_eval(tune, recipe)
        row = {"checkpoint": tag, "path": str(path), "decoder": recipe, "metrics": metrics}
        search_results.append(row); predictions_by_candidate[(tag,recipe["name"])] = preds
        print(tag, recipe["name"], metrics)
# Primary-metric guard; then optimize BLEU as requested.
original_rows = [r for r in search_results if r["checkpoint"] == "original"]
bert_floor = max(r["metrics"]["bertscore_f1"] for r in original_rows) - MAX_BERT_REGRESSION
eligible_rows = [r for r in search_results if r["metrics"]["bertscore_f1"] >= bert_floor]
best_original = max([r for r in eligible_rows if r["checkpoint"] == "original"], key=lambda r:r["metrics"]["bleu"])
winner = max(eligible_rows, key=lambda r:r["metrics"]["bleu"])
if winner["metrics"]["bleu"] <= best_original["metrics"]["bleu"] + MIN_BLEU_GAIN:
    winner = best_original
load_weights(winner["path"])
FINAL_DIR = OUT / "final_model"; FINAL_DIR.mkdir(exist_ok=True)
save_model(FINAL_DIR / "model.pt", winner)
tokenizer.save_pretrained(str(FINAL_DIR))
dump_json(FINAL_DIR / "config.json", ARCH)
dump_json(FINAL_DIR / "generation_config.json", winner["decoder"])
dump_json(OUT / "decoder_search.json", search_results)
dump_json(OUT / "selected_run.json", winner)
print("LOCKED CHOICE:", winner)


greedy:   0%|          | 0/64 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


original greedy {'bertscore_f1': 0.8124821782112122, 'rouge_l_f1': 0.20049247914309953, 'bleu': 0.05660556574904561, 'mean_words': 127.453125, 'exact_128_words': 63}


greedy_r105:   0%|          | 0/64 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


original greedy_r105 {'bertscore_f1': 0.8138529658317566, 'rouge_l_f1': 0.19384765625, 'bleu': 0.05524042977063774, 'mean_words': 128.0, 'exact_128_words': 64}


... (8 outputs left out)

finetuned greedy_r105 {'bertscore_f1': 0.8184337019920349, 'rouge_l_f1': 0.2000732421875, 'bleu': 0.05524611681696817, 'mean_words': 128.0, 'exact_128_words': 64}


original_sampling:   0%|          | 0/64 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


finetuned original_sampling {'bertscore_f1': 0.8302903771400452, 'rouge_l_f1': 0.1988525390625, 'bleu': 0.05840555473458334, 'mean_words': 128.0, 'exact_128_words': 64}
LOCKED CHOICE: {'checkpoint': 'original', 'path': '/kaggle/input/datasets/paveetsaluja/capstone-m4-ckpts/best.pt', 'decoder': {'name': 'original_sampling', 'sample': True, 'temperature': 0.5, 'top_p': 0.9, 'top_k': 50, 'rep': 1.1}, 'metrics': {'bertscore_f1': 0.8257151246070862, 'rouge_l_f1': 0.196533203125, 'bleu': 0.0598691953497314, 'mean_words': 128.0, 'exact_128_words': 64}}


In [13]:
# Audit is never consulted by training or the selection logic.
load_weights(CKPT_PATH)
audit_base_metrics, audit_base = generation_eval(audit, best_original["decoder"])
load_weights(FINAL_DIR / "model.pt")
audit_final_metrics, audit_final = generation_eval(audit, winner["decoder"])
references = [r["reference"] for r in audit]
rng = np.random.default_rng(1984)
deltas = []
for _ in range(1000):
    ix = rng.integers(0, len(audit), len(audit))
    refs = [[references[i] for i in ix]]
    ref_tokens = [references[i].split() for i in ix]
    final_bleu = official.bleu4([audit_final[i].split() for i in ix], ref_tokens)
    base_bleu = official.bleu4([audit_base[i].split() for i in ix], ref_tokens)
    deltas.append(final_bleu-base_bleu)
audit_report = {"original_tuned": audit_base_metrics, "selected": audit_final_metrics,
                "bleu_delta": audit_final_metrics["bleu"]-audit_base_metrics["bleu"],
                "paired_bootstrap_delta_95pct": [float(x) for x in np.percentile(deltas,[2.5,97.5])],
                "n_documents": len(audit), "selection_used_audit": False}
dump_json(OUT / "audit_report.json", audit_report)
dump_json(OUT / "audit_predictions.json", [dict(r, original=a, selected=b) for r,a,b in zip(audit,audit_base,audit_final)])
print(json.dumps(audit_report, indent=2))
# Export directly usable input to the exact attached evaluator.
heldout_truth = {str(i): {"text": r["reference"]} for i,r in enumerate(audit)}
heldout_prediction = {str(i): {"text": p} for i,p in enumerate(audit_final)}
official.validate(heldout_truth, "truth"); official.validate(heldout_prediction, "prediction")
dump_json(OUT / "heldout_truth.json", heldout_truth)
dump_json(OUT / "heldout_prediction.json", heldout_prediction)
print("Standalone parity command:", sys.executable, EVALUATOR_PATH,
      "--truth", OUT / "heldout_truth.json", "--prediction", OUT / "heldout_prediction.json")


original_sampling:   0%|          | 0/64 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


original_sampling:   0%|          | 0/64 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{
  "original_tuned": {
    "bertscore_f1": 0.8302745819091797,
    "rouge_l_f1": 0.2076416015625,
    "bleu": 0.06351133369323773,
    "mean_words": 128.0,
    "exact_128_words": 64
  },
  "selected": {
    "bertscore_f1": 0.8302745819091797,
    "rouge_l_f1": 0.2076416015625,
    "bleu": 0.06351133369323773,
    "mean_words": 128.0,
    "exact_128_words": 64
  },
  "bleu_delta": 0.0,
  "paired_bootstrap_delta_95pct": [
    0.0,
    0.0
  ],
  "n_documents": 64,
  "selection_used_audit": false
}
Standalone parity command: /usr/bin/python3 /kaggle/working/m4_improved/evaluate_phase1.py --truth /kaggle/working/m4_improved/heldout_truth.json --prediction /kaggle/working/m4_improved/heldout_prediction.json


## 7. Generate and strictly validate the submission

The document names `Phase1- Test.json`, whereas your earlier notebook uses `test.json`. Resolve the actual file at the pinned revision rather than guessing. If multiple test files exist, specify `TEST_FILE_OVERRIDE`.

Every ID is preserved verbatim. Each output must contain **exactly 128 words**, with continuation only. The archive is created only after these checks pass. A failed length check saves drafts and diagnostics; it does not silently pad or truncate a short output. No test text is used for training or checkpoint selection.


In [14]:
(OUT/"submission.zip").unlink(missing_ok=True)
repo_files=HfApi().list_repo_files(HF_REPO,repo_type="dataset",revision=revision,token=HF_TOKEN)
test_candidates=[f for f in repo_files if f.startswith("Phase_1_Pretraining/Test/") and f.lower().endswith(".json")]
if TEST_FILE_OVERRIDE is not None:
    assert TEST_FILE_OVERRIDE in test_candidates
    test_file=TEST_FILE_OVERRIDE
elif len(test_candidates)==1:
    test_file=test_candidates[0]
else:
    raise ValueError(f"Set TEST_FILE_OVERRIDE to one of {test_candidates}")
test_path=hf_hub_download(HF_REPO,test_file,repo_type="dataset",revision=revision,token=HF_TOKEN)
with open(test_path,encoding="utf-8") as f:test_data=json.load(f)
assert isinstance(test_data,dict) and test_data
assert all(isinstance(k,str) and isinstance(v.get("text"),str) for k,v in test_data.items())
load_weights(FINAL_DIR/"model.pt")
predictions={};diagnostics=[]
for i,(eid,rec) in enumerate(tqdm(test_data.items(),desc="Submission")):
    predictions[eid]={"text":predict_one(rec["text"],winner["decoder"],SEED+i)}
    diagnostics.append({"id":eid,"prompt_words":len(rec["text"].split()),
                        "prompt_tokens":len(tokenizer.encode(rec["text"],add_special_tokens=False)),
                        "output_words":len(predictions[eid]["text"].split())})
dump_json(OUT/"prediction_draft.json",predictions)
dump_json(OUT/"submission_diagnostics.json",diagnostics)
assert set(predictions)==set(test_data)
bad=[r for r in diagnostics if r["output_words"]!=TARGET_WORDS]
assert not bad, f"No ZIP created: {len(bad)} outputs are not 128 words; inspect submission_diagnostics.json"
dump_json(OUT/"prediction.json",predictions)
with zipfile.ZipFile(OUT/"submission.zip","w",compression=zipfile.ZIP_DEFLATED) as z:
    z.write(OUT/"prediction.json",arcname="prediction.json")
with zipfile.ZipFile(OUT/"submission.zip") as z:assert z.namelist()==["prediction.json"]
print("Submission:",OUT/"submission.zip")
print("Test prompt word-count mismatches:",sum(r["prompt_words"]!=PROMPT_WORDS for r in diagnostics))


test.json: 0.00B [00:00, ?B/s]

Submission:   0%|          | 0/1000 [00:00<?, ?it/s]

Submission: /kaggle/working/m4_improved/submission.zip
Test prompt word-count mismatches: 0


## 8. Files to retain

- **`last_finetuned.pt`**: actual trained weights saved at the latest evaluation boundary, regardless of selection.
- **`cpt_final.pt` / `sft_final.pt`**: actual weights at the end of each training stage.
- **`best.pt`**: best qualifying training checkpoint under fixed greedy decoding; can remain original if no candidate qualifies.
- **`final_model.zip`**: selected deployment model and tokenizer, after the decoder comparison.
- **`submission.zip`**: selected-model predictions, only if every output has 128 words and IDs match exactly.
- **`audit_report.json`**, **`training_log.json`**, **`decoder_search.json`**: before/after evidence using the attached evaluator definitions. The audit split never drives selection.
- **`evaluate_phase1.py`**, **`heldout_truth.json`**, **`heldout_prediction.json`**, **`metric_environment.json`**: exact standalone evaluator, its inputs and environment record.

No gains are fabricated. Training can change the weights without improving the held-out metrics. The original model remains a candidate so a failed experiment need not replace it. This is still one M4 architecture, not four distinct architectures for the course requirement. Your report's interpretation and AI-use declaration remain yours.

Provenance: your original notebooks, attached *Capstone Document.md* and attached *evaluate_phase1.py*. Technical references: [HF streaming](https://huggingface.co/docs/huggingface_hub/package_reference/hf_file_system) and [BERTScore](https://github.com/Tiiiger/bert_score).


In [15]:
MODEL_DEFINITION = 'import math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.nn as nn\nimport torch.nn.functional as F\nclass RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-6):\n        super().__init__()\n        self.eps = eps\n        self.weight = nn.Parameter(torch.ones(dim))\n    def forward(self, x):\n        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)\n        return (x.float() * rms).type_as(x) * self.weight\n\ndef precompute_rope_freqs(dim, max_seq_len, base=10000.0):\n    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))\n    t = torch.arange(max_seq_len, dtype=torch.float32)\n    freqs = torch.outer(t, freqs)\n    return torch.cos(freqs), torch.sin(freqs)\n\ndef apply_rope(x, freqs_cos, freqs_sin):\n    x_r = x.reshape(*x.shape[:-1], -1, 2)\n    x0, x1 = x_r[..., 0], x_r[..., 1]\n    seq_len = x.shape[2]\n    cos = freqs_cos[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)\n    sin = freqs_sin[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)\n    out0 = x0 * cos - x1 * sin\n    out1 = x0 * sin + x1 * cos\n    return torch.stack([out0, out1], dim=-1).reshape_as(x)\n\nclass SwiGLU_FFN(nn.Module):\n    def __init__(self, d_model, d_ff, dropout=0.0):\n        super().__init__()\n        self.w_gate = nn.Linear(d_model, d_ff, bias=False)\n        self.w_up = nn.Linear(d_model, d_ff, bias=False)\n        self.w_down = nn.Linear(d_ff, d_model, bias=False)\n        self.dropout = nn.Dropout(dropout)\n    def forward(self, x):\n        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))\n\ndef repeat_kv(x, n_rep):\n    """Repeat KV heads to match query heads. x: (B, n_kv_heads, T, D) -> (B, n_heads, T, D)"""\n    if n_rep == 1: return x\n    B, H, T, D = x.shape\n    return x[:, :, None, :, :].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)\n\nclass GQAttention(nn.Module):\n    """Grouped Query Attention with optional sliding window."""\n    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, dropout=0.0, use_window=False, window_size=256):\n        super().__init__()\n        self.n_heads = n_heads; self.n_kv_heads = n_kv_heads\n        self.head_dim = head_dim; self.n_rep = n_heads // n_kv_heads\n        self.use_window = use_window; self.window_size = window_size\n        self.attn_p = dropout\n        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)\n        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)\n        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)\n        self.out_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)\n        self.resid_dropout = nn.Dropout(dropout)\n    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):\n        B, T, _ = x.shape\n        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)\n        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)\n        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)\n        q = apply_rope(q, freqs_cos, freqs_sin)\n        k = apply_rope(k, freqs_cos, freqs_sin)\n        if past_kv is not None:\n            k = torch.cat([past_kv[0], k], dim=2)\n            v = torch.cat([past_kv[1], v], dim=2)\n        new_kv = (k, v) if use_cache else None   # cache stores only n_kv_heads (the GQA memory saving)\n        dp = self.attn_p if self.training else 0.0\n        if past_kv is not None:\n            # Cached single-token decode. Window layers only look at the last `window_size` keys,\n            # matching the training-time sliding-window mask.\n            if self.use_window:\n                k = k[:, :, -self.window_size:]; v = v[:, :, -self.window_size:]\n            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)\n            out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp)\n        else:\n            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)\n            if self.use_window and T > 1:\n                # Boolean sliding-window causal mask (True = may attend)\n                row = torch.arange(T, device=x.device).unsqueeze(1)\n                col = torch.arange(T, device=x.device).unsqueeze(0)\n                mask = (col <= row) & (col > row - self.window_size)\n                out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp)\n            else:\n                out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dp)\n        out = out.transpose(1, 2).contiguous().view(B, T, -1)\n        return self.resid_dropout(self.out_proj(out)), new_kv\n\nclass TransformerBlock(nn.Module):\n    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout=0.0,\n                 use_window=False, window_size=256):\n        super().__init__()\n        self.attn_norm = RMSNorm(d_model)\n        self.attn = GQAttention(d_model, n_heads, n_kv_heads, head_dim, dropout, use_window, window_size)\n        self.ffn_norm = RMSNorm(d_model)\n        self.ffn = SwiGLU_FFN(d_model, d_ff, dropout)\n    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):\n        attn_out, new_kv = self.attn(self.attn_norm(x), freqs_cos, freqs_sin, use_cache, past_kv)\n        x = x + attn_out\n        x = x + self.ffn(self.ffn_norm(x))\n        return x, new_kv\n\nfrom torch.utils.checkpoint import checkpoint as grad_checkpoint\nclass EfficientLegalLM(nn.Module):\n    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, head_dim, d_ff,\n                 max_seq_len, rope_base=10000.0, dropout=0.0, window_size=256, use_gc=True):\n        super().__init__()\n        self.d_model = d_model; self.n_layers = n_layers; self.vocab_size = vocab_size\n        self.max_seq_len = max_seq_len; self.use_gc = use_gc\n        self.tok_emb = nn.Embedding(vocab_size, d_model)\n        self.emb_dropout = nn.Dropout(dropout)\n        # Alternating layers: even = sliding window, odd = full attention\n        self.layers = nn.ModuleList([\n            TransformerBlock(d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout,\n                use_window=(i % 2 == 0), window_size=window_size)\n            for i in range(n_layers)\n        ])\n        self.final_norm = RMSNorm(d_model)\n        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)\n        self.lm_head.weight = self.tok_emb.weight\n        fc, fs = precompute_rope_freqs(head_dim, max_seq_len, rope_base)\n        self.register_buffer("freqs_cos", fc, persistent=False)\n        self.register_buffer("freqs_sin", fs, persistent=False)\n        self._init_weights()\n    def _init_weights(self):\n        for m in self.modules():\n            if isinstance(m, nn.Linear):\n                torch.nn.init.normal_(m.weight, std=0.02)\n                if m.bias is not None: torch.nn.init.zeros_(m.bias)\n            elif isinstance(m, nn.Embedding):\n                torch.nn.init.normal_(m.weight, std=0.02)\n        for layer in self.layers:\n            s = 0.02 / math.sqrt(2 * self.n_layers)\n            torch.nn.init.normal_(layer.attn.out_proj.weight, std=s)\n            torch.nn.init.normal_(layer.ffn.w_down.weight, std=s)\n    def forward(self, input_ids, targets=None, use_cache=False, past_kvs=None):\n        B, T = input_ids.shape\n        x = self.emb_dropout(self.tok_emb(input_ids))\n        past_len = past_kvs[0][0].shape[2] if past_kvs is not None else 0\n        fc = self.freqs_cos[past_len:past_len+T]\n        fs = self.freqs_sin[past_len:past_len+T]\n        presents = [] if use_cache else None\n        for i, layer in enumerate(self.layers):\n            pkv = past_kvs[i] if past_kvs is not None else None\n            if self.use_gc and self.training:\n                x, _ = grad_checkpoint(layer, x, fc, fs, False, None, use_reentrant=False)\n            else:\n                x, kv = layer(x, fc, fs, use_cache, pkv)\n                if use_cache: presents.append(kv)\n        x = self.final_norm(x)\n        logits = self.lm_head(x)\n        loss = None\n        if targets is not None:\n            loss = F.cross_entropy(logits[:, :-1, :].contiguous().view(-1, self.vocab_size),\n                targets[:, 1:].contiguous().view(-1), ignore_index=-100)\n        return logits, loss, presents\n    @torch.no_grad()\n    def generate(self, input_ids, max_new_tokens, temperature=0.7, top_p=0.9,\n                 top_k=50, repetition_penalty=1.1, eos_token_id=None):\n        self.eval(); B = input_ids.shape[0]\n        generated = input_ids; past_kvs = None\n        for _ in range(max_new_tokens):\n            curr = generated[:, -1:] if past_kvs is not None else generated\n            logits, _, past_kvs = self.forward(curr, use_cache=True, past_kvs=past_kvs)\n            logits = logits[:, -1, :].float()\n            if repetition_penalty != 1.0:\n                for b in range(B):\n                    idx = torch.unique(generated[b])\n                    sel = logits[b, idx]\n                    logits[b, idx] = torch.where(sel > 0, sel / repetition_penalty, sel * repetition_penalty)\n            logits = logits / temperature\n            if top_k > 0:\n                topk_v, _ = torch.topk(logits, min(top_k, logits.size(-1)))\n                logits[logits < topk_v[:, -1:]] = float(\'-inf\')\n            if top_p < 1.0:\n                sl, si = torch.sort(logits, descending=True)\n                cp = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)\n                rm = cp > top_p; rm[:, 1:] = rm[:, :-1].clone(); rm[:, 0] = False\n                for b in range(B): logits[b, si[b][rm[b]]] = float(\'-inf\')\n            probs = F.softmax(logits, dim=-1)\n            next_tok = torch.multinomial(probs, num_samples=1)\n            generated = torch.cat([generated, next_tok], dim=1)\n            if eos_token_id is not None and (next_tok == eos_token_id).all(): break\n        return generated\n'
(FINAL_DIR / "model_definition.py").write_text(MODEL_DEFINITION)
# Usage: from model_definition import EfficientLegalLM
# ck = torch.load('model.pt', map_location='cpu', weights_only=True)
# m = EfficientLegalLM(**ck['config']); m.load_state_dict(ck['model_state_dict'], strict=True)
shutil.make_archive(str(OUT / "final_model"), "zip", FINAL_DIR)
print("Model bundle:", OUT / "final_model.zip")
print("Audit report:", OUT / "audit_report.json")


Model bundle: /kaggle/working/m4_improved/final_model.zip
Audit report: /kaggle/working/m4_improved/audit_report.json
